In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemError(
        "⚠️ No GPU found. In Colab, go to: Runtime → Change runtime type → Hardware accelerator → GPU."
    )

device = "cuda"
print("Using device:", device)

!pip install -q transformers accelerate bitsandbytes

import torch
import math
import time
import psutil
import os

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

model_id = "gpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --- Memory helpers ---

def get_process_memory_mb():
    """Return current process memory (CPU) in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2

def get_gpu_memory_mb():
    """Return current GPU memory in MB (if CUDA)."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1024**2

def describe_memory(label):
    cpu_mem = get_process_memory_mb()
    gpu_mem = get_gpu_memory_mb()
    print(f"[{label}] CPU memory: {cpu_mem:8.2f} MB   |   GPU memory: {gpu_mem:8.2f} MB")


# --- Perplexity helper ---

@torch.no_grad()
def compute_perplexity(model, tokenizer, text: str) -> float:
    model.eval()
    enc = tokenizer(text, return_tensors="pt").to(device)
    outputs = model(**enc, labels=enc["input_ids"])
    loss = outputs.loss.item()
    return math.exp(loss)


# --- Generation helper ---

@torch.no_grad()
def timed_generate(model, tokenizer, prompt: str, max_new_tokens: int = 40, num_runs: int = 3):
    model.eval()
    times = []
    last_output = None

    for i in range(num_runs):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        torch.cuda.empty_cache()
        start = time.perf_counter()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize(device) if device == "cuda" else None
        end = time.perf_counter()
        times.append(end - start)
        last_output = tokenizer.decode(out[0], skip_special_tokens=True)

    avg_time = sum(times) / len(times)
    return avg_time, last_output


Using device: cuda
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.3 MB/s eta 0:00:00
Device: cuda


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer loaded:", model_id)

# Clear GPU before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading baseline FP16 model...")
describe_memory("Before FP16 load")

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device)

describe_memory("After FP16 load")
print("Model dtype:", next(model_fp16.parameters()).dtype)
print("Approx model footprint (reported by HF):", model_fp16.get_memory_footprint() / 1024**2, "MB")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Tokenizer loaded: gpt2
Loading baseline FP16 model...
[Before FP16 load] CPU memory:   811.58 MB   |   GPU memory:     0.00 MB


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[After FP16 load] CPU memory:  1759.70 MB   |   GPU memory:   255.49 MB
Model dtype: torch.float16
Approx model footprint (reported by HF): 249.3501205444336 MB


In [ ]:
# Clear GPU before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading 8-bit quantized model (bitsandbytes LLM.int8)...")
describe_memory("Before 8-bit load")

bnb_8bit_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,           # default from LLM.int8() paper
    llm_int8_enable_fp32_cpu_offload=False,
)

model_8bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",                # let Accelerate map layers to GPU
    quantization_config=bnb_8bit_config,
    torch_dtype=torch.float16,        # non-linear layers remain in this dtype
)

describe_memory("After 8-bit load")
print("8-bit model type:", type(model_8bit))
try:
    print("Approx 8-bit footprint:", model_8bit.get_memory_footprint() / 1024**2, "MB")
except Exception as e:
    print("get_memory_footprint not available:", e)


Loading 8-bit quantized model (bitsandbytes LLM.int8)...
[Before 8-bit load] CPU memory:  1743.29 MB   |   GPU memory:   255.49 MB
[After 8-bit load] CPU memory:  2050.17 MB   |   GPU memory:   425.42 MB
8-bit model type: <class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>
Approx 8-bit footprint: 168.3501205444336 MB


In [ ]:
# Clear GPU before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Loading 4-bit NF4 quantized model...")
describe_memory("Before 4-bit load")

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # compute in fp16 for speed
    bnb_4bit_quant_type="nf4",             # NormalFloat4, good for Gaussian weights
    bnb_4bit_use_double_quant=True,        # nested quantization to compress scales
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_4bit_config,
    torch_dtype=torch.float16,
)

describe_memory("After 4-bit load")
try:
    print("Approx 4-bit footprint:", model_4bit.get_memory_footprint() / 1024**2, "MB")
except Exception as e:
    print("get_memory_footprint not available:", e)


Loading 4-bit NF4 quantized model...
[Before 4-bit load] CPU memory:  2050.17 MB   |   GPU memory:   425.42 MB
[After 4-bit load] CPU memory:  2234.61 MB   |   GPU memory:   555.73 MB
Approx 4-bit footprint: 127.8501205444336 MB


In [ ]:
sample_text = (
    "Quantization allows us to run large language models on smaller hardware "
    "by reducing the precision of the weights and activations."
)

print("Sample text:", sample_text)
print()

ppl_fp16 = compute_perplexity(model_fp16, tokenizer, sample_text)
print(f"Perplexity (FP16 baseline): {ppl_fp16:.3f}")

ppl_8bit = compute_perplexity(model_8bit, tokenizer, sample_text)
print(f"Perplexity (8-bit):         {ppl_8bit:.3f}")

ppl_4bit = compute_perplexity(model_4bit, tokenizer, sample_text)
print(f"Perplexity (4-bit NF4):     {ppl_4bit:.3f}")


Sample text: Quantization allows us to run large language models on smaller hardware by reducing the precision of the weights and activations.



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Perplexity (FP16 baseline): 67.668
Perplexity (8-bit):         68.204
Perplexity (4-bit NF4):     73.950


In [ ]:
prompt = "In the future, efficient AI systems will"

print("Prompt:", prompt)
print("\n--- FP16 baseline generation ---")
t_fp16, out_fp16 = timed_generate(model_fp16, tokenizer, prompt)
print(f"Average time (FP16): {t_fp16:.3f} s")
print(out_fp16)

print("\n--- 8-bit generation ---")
t_8bit, out_8bit = timed_generate(model_8bit, tokenizer, prompt)
print(f"Average time (8-bit): {t_8bit:.3f} s")
print(out_8bit)

print("\n--- 4-bit NF4 generation ---")
t_4bit, out_4bit = timed_generate(model_4bit, tokenizer, prompt)
print(f"Average time (4-bit): {t_4bit:.3f} s")
print(out_4bit)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: In the future, efficient AI systems will

--- FP16 baseline generation ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Average time (FP16): 1.858 s
In the future, efficient AI systems will be able to do things like search for and find people, and to do things like search for and find people, and to do things like search for and find people, and to do things like search

--- 8-bit generation ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Average time (8-bit): 6.018 s
In the future, efficient AI systems will be able to do things like search for and find people, and to search for and find people who are in the same place.

The AI system will be able to do things like search for

--- 4-bit NF4 generation ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Average time (4-bit): 2.202 s
In the future, efficient AI systems will be able to do things like search for information, search for information, and search for information.

The AI system will be able to do things like search for information, search for information, and
